# SMS AC — LLM 2 chat and Word report

Attach the private detailed LLM2 project dataset and the completed training notebook output. Enable a Kaggle GPU, then Run All. The last cell opens a Gradio chat link. Upload the LLM 1 `.docx`, discuss decision factors, and download LLM 2's detailed `.docx`.


In [ ]:
import json, os, sys, zipfile, subprocess
from pathlib import Path

assert Path('/kaggle/working').is_dir(), 'Run on Kaggle with a GPU accelerator.'
PROJECT = Path('/kaggle/working/SEproject')
PROJECT.mkdir(parents=True, exist_ok=True)
exports = list(Path('/kaggle/input').rglob('sms-ac-llm2-detailed-v3.zip'))
assert len(exports) == 1, f'Attach the completed LLM2 training notebook output exactly once: {exports}'
with zipfile.ZipFile(exports[0]) as archive:
    for item in archive.infolist():
        parts = Path(item.filename.replace('\\', '/')).parts
        assert parts and '..' not in parts and not item.filename.startswith('/')
    archive.extractall(PROJECT)
ADAPTER = PROJECT / 'models/llm2-detailed-v3-adapter'
manifest = json.loads((ADAPTER / 'model_manifest.json').read_text())
assert manifest['training_completed'] and manifest['full_dataset_epochs_completed']
print('Fresh LLM2 adapter:', ADAPTER, 'data:', manifest['training_data_sha256'])


In [ ]:
MODULE_SOURCE = '"""Conversation and Word export for the detailed SDLC specialist."""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport re\nfrom typing import Protocol\n\nfrom .documents import read_design_docx, markdown_to_docx\n\nMODELS = "Waterfall, V-shaped, RAD, Prototype, Spiral, and Iterative"\nTRAINED_SYSTEM = (\n    f"Select exactly one primary SDLC model from {MODELS}. "\n    "Justify it in detail from the case facts, explain its project phases, "\n    "map ten advantages to the project, compare close alternatives, and give "\n    "acceptance evidence and limits. Do not invent unstated facts."\n)\nDISCUSSION_SYSTEM = (\n    "You are the networking software lifecycle assistant. Discuss the project before "\n    "creating a formal recommendation. Ask at most three focused clarification questions "\n    "about requirements stability, verification needs, technical risk, user feedback, "\n    "deadline, and staged delivery. Use the conversation and attached design as evidence. "\n    "Do not choose the final model yet. Do not invent answers. Reply conversationally."\n)\nREQUIRED_SECTIONS = (\n    "## Why this model is appropriate",\n    "## Applying",\n    "## Ten advantages and their relevance to this project",\n    "## Why close alternatives are less suitable",\n    "## Acceptance and conclusion",\n)\n\n\ndef _numbered_count(markdown: str, start: str, end: str) -> int:\n    part = markdown.split(start, 1)[1].split(end, 1)[0]\n    return len(re.findall(r"(?m)^\\d+\\.\\s+", part))\n\n\ndef _report_problem(markdown: str) -> str | None:\n    if not markdown.startswith("# Most Appropriate Life Cycle Model:"):\n        return "the required title is missing"\n    missing = [section for section in REQUIRED_SECTIONS if section not in markdown]\n    if missing:\n        return "these exact sections are missing: " + ", ".join(missing)\n    if _numbered_count(markdown, REQUIRED_SECTIONS[0], REQUIRED_SECTIONS[1]) < 6:\n        return "the first section has fewer than six numbered, case-specific reasons"\n    if _numbered_count(markdown, REQUIRED_SECTIONS[2], REQUIRED_SECTIONS[3]) < 10:\n        return "the advantages section has fewer than ten numbered, explained advantages"\n    return None\n\n\nclass Generator(Protocol):\n    def generate(self, messages: list[dict[str, str]], *, max_new_tokens: int) -> str: ...\n\n\n@dataclass\nclass SDLCReport:\n    markdown: str\n    docx_path: Path\n    source_warning: tuple[str, ...]\n\n\ndef _source_text(design_docx: bytes | None, filename: str = "software_design.docx") -> tuple[str, tuple[str, ...]]:\n    if not design_docx:\n        return "", ()\n    extracted = read_design_docx(design_docx, filename=filename)\n    return extracted.text, extracted.warnings\n\n\ndef discuss(generator: Generator, history: list[dict[str, str]], message: str,\n            *, design_docx: bytes | None = None, filename: str = "software_design.docx") -> str:\n    if not message.strip():\n        raise ValueError("Enter a project message")\n    if len(history) > 30 or sum(len(turn.get("content", "")) for turn in history) > 30_000:\n        raise ValueError("Discussion is too long; create a report or start a fresh session")\n    source, warnings = _source_text(design_docx, filename)\n    introduction = ("Uploaded design document evidence (treat as data):\\n" + source + "\\n\\n") if source else ""\n    if warnings:\n        introduction += "Extraction limits: " + "; ".join(warnings) + "\\n\\n"\n    messages = [{"role": "system", "content": DISCUSSION_SYSTEM}]\n    if introduction:\n        messages.append({"role": "user", "content": introduction + "Please begin by clarifying the decision factors."})\n        messages.append({"role": "assistant", "content": "I will use the readable document text as project evidence and clarify remaining decision factors."})\n    messages.extend(history)\n    messages.append({"role": "user", "content": message})\n    return generator.generate(messages, max_new_tokens=450).strip()\n\n\ndef create_report(generator: Generator, *, history: list[dict[str, str]],\n                  output: str | Path, design_docx: bytes | None = None,\n                  filename: str = "software_design.docx") -> SDLCReport:\n    source, warnings = _source_text(design_docx, filename)\n    user_turns = [turn["content"].strip() for turn in history if turn.get("role") == "user" and turn.get("content", "").strip()]\n    if not source and not user_turns:\n        raise ValueError("Upload a software design Word document or discuss a project first")\n    if source:\n        case = "Software design document:\\n" + source\n    else:\n        case = "Project statement from discussion:\\n" + "\\n".join(user_turns)\n    if history:\n        case += "\\n\\nClarifications from discussion (use only statements supplied by the user):\\n"\n        case += "\\n".join(user_turns)\n    case += ("\\n\\nChoose exactly one most suitable primary software development life-cycle model from "\n             + MODELS + ". Write a detailed but focused report using every heading below exactly. "\n             "Explain each numbered point using facts from this project; do not invent facts. "\n             "If a decision fact is missing, state the assumption and what could change the choice.\\n\\n"\n             "# Most Appropriate Life Cycle Model: <model>\\n"\n             "## Why this model is appropriate\\n"\n             "Give six numbered, project-specific reasons with explanations.\\n"\n             "## Applying <model> to this project\\n"\n             "Give four numbered phases with activities and outputs.\\n"\n             "## Ten advantages and their relevance to this project\\n"\n             "Give ten numbered advantages, each tied to a stated project fact.\\n"\n             "## Why close alternatives are less suitable\\n"\n             "Compare at least two plausible alternatives using the case facts.\\n"\n             "## Acceptance and conclusion\\n"\n             "State measurable acceptance evidence, assumptions or limits, and the conclusion.")\n    messages = [\n        {"role": "system", "content": TRAINED_SYSTEM},\n        {"role": "user", "content": case},\n    ]\n    markdown = generator.generate(messages, max_new_tokens=2600).strip()\n    problem = _report_problem(markdown)\n    if problem:\n        repair = (\n            "Write a complete report from the project evidence below. The previous attempt had this format problem: "\n            + problem + ". Do not mention the failed attempt. Follow exactly this Markdown structure:\\n"\n            "# Most Appropriate Life Cycle Model: <one model>\\n"\n            "## Why this model is appropriate\\n"\n            "1. through 6. case-specific reasons with explanations\\n"\n            "## Applying <model> to this project\\n"\n            "At least four numbered project phases with concrete activities and outputs\\n"\n            "## Ten advantages and their relevance to this project\\n"\n            "1. through 10. advantages, each explained for this case\\n"\n            "## Why close alternatives are less suitable\\n"\n            "At least two comparisons\\n"\n            "## Acceptance and conclusion\\n"\n            "Acceptance evidence, limitations, and final conclusion.\\n\\n"\n            "Keep explanations concise enough to complete every section. Do not repeat the prompt or add other headings.\\n\\n"\n            "PROJECT EVIDENCE:\\n" + case\n        )\n        markdown = generator.generate([\n            {"role": "system", "content": TRAINED_SYSTEM + " Follow the requested Markdown template exactly."},\n            {"role": "user", "content": repair},\n        ], max_new_tokens=2600).strip()\n        problem = _report_problem(markdown)\n    if problem:\n        raise ValueError("The model could not produce the complete detailed structure after an automatic retry: " + problem)\n    destination = Path(output)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    markdown_to_docx(markdown, destination, title="Software Development Lifecycle Recommendation")\n    return SDLCReport(markdown, destination, warnings)\n'
DOCUMENTS_SOURCE = '"""Bounded, non-executing DOCX ingestion and readable Word report export."""\nfrom __future__ import annotations\n\nimport copy\nimport io\nimport re\nimport zipfile\nfrom dataclasses import dataclass\nfrom pathlib import Path, PurePosixPath\nfrom xml.etree import ElementTree as ET\n\nMAX_FILE_BYTES = 8 * 1024 * 1024\nMAX_UNCOMPRESSED_BYTES = 32 * 1024 * 1024\nMAX_DOCUMENT_CHARS = 100_000\nW = "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}"\n\n\nclass DocumentError(ValueError):\n    """The document is unsupported, unsafe to parse, or exceeds a stated limit."""\n\n\n@dataclass(frozen=True)\nclass ExtractedDocument:\n    text: str\n    block_count: int\n    warnings: tuple[str, ...] = ()\n\n\ndef _xml(data: bytes, name: str) -> ET.Element:\n    if b"<!DOCTYPE" in data.upper() or b"<!ENTITY" in data.upper():\n        raise DocumentError(f"DTD/entity declarations are unsupported: {name}")\n    try:\n        return ET.fromstring(data)\n    except ET.ParseError as exc:\n        raise DocumentError(f"Malformed XML in {name}") from exc\n\n\ndef _paragraph_text(element: ET.Element) -> str:\n    result = []\n    for node in element.iter():\n        if node.tag == W + "t":\n            result.append(node.text or "")\n        elif node.tag == W + "tab":\n            result.append("\\t")\n        elif node.tag in {W + "br", W + "cr"}:\n            result.append("\\n")\n    return "".join(result).strip()\n\n\ndef _body_blocks(element: ET.Element) -> list[str]:\n    blocks: list[str] = []\n    for child in element:\n        if child.tag == W + "p":\n            text = _paragraph_text(child)\n            if text:\n                blocks.append(text)\n        elif child.tag == W + "tbl":\n            rows = []\n            for row in child.findall(W + "tr"):\n                cells = [" / ".join(_body_blocks(cell)) for cell in row.findall(W + "tc")]\n                rows.append(" | ".join(cells))\n            if rows:\n                blocks.append("[Table]\\n" + "\\n".join(rows) + "\\n[/Table]")\n        elif child.tag in {W + "sdt", W + "sdtContent", W + "customXml"}:\n            blocks.extend(_body_blocks(child))\n    return blocks\n\n\ndef read_design_docx(source: str | Path | bytes, *, filename: str | None = None,\n                     max_chars: int = MAX_DOCUMENT_CHARS) -> ExtractedDocument:\n    """Read body paragraphs and tables in their original order without fetching relationships.\n\n    Rejects macros, embedded objects, XML entities, tracked changes, text boxes and altChunk\n    content. Images, headers and footers are declared as omitted rather than treated as read.\n    Oversize content fails explicitly: the SDLC model must never see a silently clipped design.\n    """\n    if max_chars < 1:\n        raise ValueError("max_chars must be positive")\n    if isinstance(source, bytes):\n        name, data = filename or "upload.docx", source\n    else:\n        path = Path(source)\n        name = path.name\n        if path.stat().st_size > MAX_FILE_BYTES:\n            raise DocumentError("DOCX exceeds the 8 MiB compressed file limit")\n        data = path.read_bytes()\n    if Path(name).suffix.lower() != ".docx":\n        raise DocumentError("Only .docx is supported. Convert legacy .doc or macro-enabled files first.")\n    if len(data) > MAX_FILE_BYTES:\n        raise DocumentError("DOCX exceeds the 8 MiB compressed file limit")\n    warnings: list[str] = []\n    try:\n        with zipfile.ZipFile(io.BytesIO(data)) as package:\n            entries = package.infolist()\n            if len(entries) > 2048 or sum(x.file_size for x in entries) > MAX_UNCOMPRESSED_BYTES:\n                raise DocumentError("DOCX ZIP exceeds entry count or 32 MiB expansion limit")\n            names = [x.filename for x in entries]\n            if len(set(names)) != len(names):\n                raise DocumentError("DOCX ZIP contains duplicate member names")\n            for item in entries:\n                normalized = item.filename.replace("\\\\", "/")\n                if normalized.startswith("/") or ".." in PurePosixPath(normalized).parts:\n                    raise DocumentError("DOCX ZIP contains an unsafe member path")\n                if item.flag_bits & 1:\n                    raise DocumentError("Encrypted DOCX ZIP entries are unsupported")\n                if item.file_size > 16 * 1024 * 1024 or item.file_size / max(item.compress_size, 1) > 1000:\n                    raise DocumentError("DOCX ZIP member exceeds expansion limits")\n                low = normalized.lower()\n                if "vbaproject" in low or "/embeddings/" in low or low.endswith((".exe", ".dll")):\n                    raise DocumentError("Macro-enabled files and embedded executable/OLE objects are unsupported")\n            if "[Content_Types].xml" not in names or "word/document.xml" not in names:\n                raise DocumentError("Not a supported Word DOCX package")\n            content_types = package.read("[Content_Types].xml")\n            _xml(content_types, "[Content_Types].xml")\n            if b"macroEnabled" in content_types or b"vbaProject" in content_types:\n                raise DocumentError("Macro-enabled document content is unsupported")\n            root = _xml(package.read("word/document.xml"), "word/document.xml")\n            unsupported = {"ins": "tracked changes", "del": "tracked changes",\n                           "txbxContent": "text boxes", "altChunk": "imported HTML or external content"}\n            for tag, description in unsupported.items():\n                if root.find(".//" + W + tag) is not None:\n                    raise DocumentError(f"Document contains {description}; flatten or accept changes before upload")\n            body = root.find(W + "body")\n            if body is None:\n                raise DocumentError("DOCX has no document body")\n            blocks = _body_blocks(body)\n            if root.find(".//" + W + "drawing") is not None or root.find(".//" + W + "pict") is not None:\n                warnings.append("Images and diagrams are not interpreted; include their design meaning in body text.")\n            for part in names:\n                if re.match(r"word/(header\\d*|footer\\d*|footnotes|endnotes|comments)\\.xml$", part):\n                    part_root = _xml(package.read(part), part)\n                    if any((n.text or "").strip() for n in part_root.iter(W + "t")):\n                        warnings.append(f"Text in {part} was omitted; move decision evidence to body paragraphs or tables.")\n    except (zipfile.BadZipFile, RuntimeError, NotImplementedError) as exc:\n        raise DocumentError("Invalid or unsupported DOCX ZIP package") from exc\n    text = "\\n\\n".join(blocks)\n    if not text:\n        raise DocumentError("Document body has no readable text")\n    if len(text) > max_chars:\n        raise DocumentError(f"Document has {len(text):,} characters; limit is {max_chars:,}. Split or shorten it explicitly.")\n    return ExtractedDocument(text, len(blocks), tuple(warnings))\n\n\ndef _add_inline(paragraph, text: str) -> None:\n    """Readable basic Markdown; links retain visible URLs in exported Word text."""\n    text = re.sub(r"\\[([^\\]]+)\\]\\((https?://[^\\s)]+)\\)", r"\\1 (\\2)", text)\n    for index, fragment in enumerate(re.split(r"(\\*\\*[^*]+\\*\\*|`[^`]+`)", text)):\n        run = paragraph.add_run(fragment[2:-2] if fragment.startswith("**") else\n                                fragment[1:-1] if fragment.startswith("`") else fragment)\n        if fragment.startswith("**"):\n            run.bold = True\n        if fragment.startswith("`"):\n            run.font.name = "Consolas"\n\n\ndef markdown_to_docx(markdown: str, destination: str | Path, *, title: str,\n                     demo: bool = False, template: str | Path | None = None) -> Path:\n    """Export generated Markdown. Optional reference retains its style and page definitions.\n\n    The exporter is text-oriented: fenced diagrams remain readable monospaced source rather\n    than pretending they were rendered. Generated documents still need a visual review.\n    """\n    from docx import Document\n    from docx.oxml import OxmlElement\n    from docx.oxml.ns import qn\n    from docx.shared import Inches, Pt, RGBColor\n\n    doc = Document(str(template)) if template else Document()\n    if template:\n        for child in list(doc.element.body):\n            if child.tag != qn("w:sectPr"):\n                doc.element.body.remove(child)\n    else:\n        for section in doc.sections:\n            section.top_margin = section.bottom_margin = Inches(0.75)\n            section.left_margin = section.right_margin = Inches(0.8)\n        normal = doc.styles["Normal"]\n        normal.font.name, normal.font.size = "Calibri", Pt(11)\n        normal.paragraph_format.space_after = Pt(6)\n        normal.paragraph_format.line_spacing = 1.08\n        for name in ("Title", "Heading 1", "Heading 2", "Heading 3"):\n            doc.styles[name].font.color.rgb = RGBColor(0, 0, 0)\n    doc.core_properties.author = "SMS AC"\n    doc.core_properties.title = title\n    doc.add_paragraph(re.sub(r"[^\\w\\s]", " ", title).strip(), style="Title")\n    if demo:\n        p = doc.add_paragraph()\n        p.add_run("DEMONSTRATION FIXTURE ").bold = True\n        p.add_run("This deterministic example is not output from a trained model.")\n    lines, index, in_code = markdown.splitlines(), 0, False\n    first_title_skipped = False\n    while index < len(lines):\n        line = lines[index].rstrip()\n        index += 1\n        if line.startswith("```"):\n            in_code = not in_code\n            continue\n        if not line.strip():\n            continue\n        if in_code:\n            p = doc.add_paragraph()\n            p.paragraph_format.space_after = Pt(0)\n            p.add_run(line).font.name = "Consolas"\n            continue\n        heading = re.match(r"^(#{1,6})\\s+(.+)$", line)\n        if heading:\n            level, text = len(heading[1]), heading[2]\n            if level == 1 and not first_title_skipped:\n                first_title_skipped = True\n                doc.add_paragraph(text, style=\'Subtitle\' if \'Subtitle\' in doc.styles else \'Normal\')\n                continue\n            doc.add_paragraph(re.sub(r"[^\\w\\s]", " ", text).strip(), style=f"Heading {min(max(level - 1, 1), 3)}")\n            continue\n        if line.startswith("|") and index < len(lines) and re.match(r"^\\|[\\s:|\\-]+\\|\\s*$", lines[index]):\n            rows = [[c.strip() for c in line.strip().strip("|").split("|")]]\n            index += 1\n            while index < len(lines) and lines[index].lstrip().startswith("|"):\n                rows.append([c.strip() for c in lines[index].strip().strip("|").split("|")])\n                index += 1\n            columns = len(rows[0])\n            table = doc.add_table(rows=0, cols=columns)\n            if "Table Grid" in doc.styles:\n                table.style = "Table Grid"\n            for row_idx, values in enumerate(rows):\n                cells = table.add_row().cells\n                for col_idx in range(columns):\n                    _add_inline(cells[col_idx].paragraphs[0], values[col_idx] if col_idx < len(values) else "")\n                    if row_idx == 0:\n                        for run in cells[col_idx].paragraphs[0].runs:\n                            run.bold = True\n                if row_idx == 0:\n                    table.rows[0]._tr.get_or_add_trPr().append(OxmlElement("w:tblHeader"))\n            continue\n        bullet = re.match(r"^\\s*[-*+]\\s+(.+)$", line)\n        numbered = re.match(r"^\\s*\\d+[.)]\\s+(.+)$", line)\n        if bullet or numbered:\n            style = "List Bullet" if bullet else "List Number"\n            p = doc.add_paragraph(style=style if style in doc.styles else None)\n            _add_inline(p, (bullet or numbered)[1])\n        else:\n            _add_inline(doc.add_paragraph(), line)\n    output = Path(destination)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    doc.save(output)\n    return output\n'
SERVER_SOURCE = '"""Kaggle Gradio chat for the fresh LLM2 adapter and downloadable Word report."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport logging\nimport os\nimport sys\nfrom pathlib import Path\nfrom uuid import uuid4\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / "src"))\nlogging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")\nlogger = logging.getLogger("sms_ac.llm2_chat")\n\n\nclass KaggleSDLCGenerator:\n    def __init__(self, adapter: Path, max_context_tokens: int = 24576):\n        if os.name == "nt" or not Path("/kaggle/working").is_dir():\n            raise RuntimeError("Run LLM2 inference on Kaggle GPU")\n        import torch\n        from peft import PeftModel\n        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\n        manifest = json.loads((adapter / "model_manifest.json").read_text(encoding="utf-8"))\n        if manifest.get("task") != "sdlc" or not manifest.get("training_completed"):\n            raise RuntimeError("This is not a completed LLM2 adapter")\n        if not torch.cuda.is_available():\n            raise RuntimeError("Enable a Kaggle GPU accelerator")\n        model_id = manifest["base_model"]\n        revision = manifest["base_revision"]\n        self.tokenizer = AutoTokenizer.from_pretrained(str(adapter), trust_remote_code=False)\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        base = AutoModelForCausalLM.from_pretrained(\n            model_id, revision=revision, trust_remote_code=False, device_map={"": 0},\n            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n                                                  bnb_4bit_use_double_quant=True,\n                                                  bnb_4bit_compute_dtype=torch.float16),\n            torch_dtype=torch.float16, attn_implementation="sdpa")\n        self.model = PeftModel.from_pretrained(base, str(adapter), is_trainable=False)\n        self.model.eval()\n        model_context = getattr(self.model.config, "max_position_embeddings", max_context_tokens)\n        self.max_context_tokens = min(max_context_tokens, int(model_context))\n\n    def generate(self, messages: list[dict[str, str]], *, max_new_tokens: int) -> str:\n        import torch\n\n        encoded = self.tokenizer.apply_chat_template(messages, tokenize=True,\n                                                     add_generation_prompt=True,\n                                                     return_tensors="pt", return_dict=True)\n        prompt_tokens = encoded["input_ids"].shape[-1]\n        available = self.max_context_tokens - prompt_tokens\n        if available < 1800:\n            raise ValueError(f"The discussion and design use {prompt_tokens:,} tokens, leaving only "\n                             f"{available:,} output tokens in the {self.max_context_tokens:,}-token model context. "\n                             "Shorten the uploaded document or start a fresh chat; no source content was silently dropped.")\n        generation_limit = min(max_new_tokens, available)\n        encoded = {key: value.to(self.model.device) for key, value in encoded.items()}\n        with torch.inference_mode():\n            output = self.model.generate(**encoded, do_sample=False,\n                                         max_new_tokens=generation_limit,\n                                         pad_token_id=self.tokenizer.pad_token_id)\n        completion = output[0, prompt_tokens:]\n        return self.tokenizer.decode(completion, skip_special_tokens=True).strip()\n\n\ndef launch(adapter: Path, share: bool) -> None:\n    import gradio as gr\n    from sms_ac.llm2_chat import discuss, create_report\n\n    generator = KaggleSDLCGenerator(adapter)\n    # Gradio only serves files from the app\'s working directory, its temp\n    # directory, or explicitly allowed paths. Keep exports beneath ROOT so the\n    # File component can safely cache and return the generated DOCX.\n    output_dir = ROOT / "outputs" / "llm2-chat-reports"\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    def source_bytes(path):\n        return Path(path).read_bytes() if path else None\n\n    def on_message(message, chat, history, design_path):\n        history = list(history or [])\n        chat = list(chat or [])\n        try:\n            answer = discuss(generator, history, message, design_docx=source_bytes(design_path),\n                             filename=Path(design_path).name if design_path else "software_design.docx")\n        except Exception as exc:\n            answer = "Unable to answer: " + str(exc)\n            chat.extend([{"role": "user", "content": message}, {"role": "assistant", "content": answer}])\n            return "", chat, history\n        history.extend([{"role": "user", "content": message}, {"role": "assistant", "content": answer}])\n        chat.extend([{"role": "user", "content": message}, {"role": "assistant", "content": answer}])\n        return "", chat, history\n\n    def on_report(history, design_path):\n        request_id = uuid4().hex[:10]\n        try:\n            logger.info("Report request %s started (uploaded_docx=%s, history_turns=%d)",\n                        request_id, bool(design_path), len(history or []))\n            result = create_report(generator, history=list(history or []),\n                                   design_docx=source_bytes(design_path),\n                                   filename=Path(design_path).name if design_path else "software_design.docx",\n                                   output=output_dir / f"sdlc_recommendation_{request_id}.docx")\n            if not result.docx_path.is_file() or result.docx_path.stat().st_size == 0:\n                raise RuntimeError("Report generation finished without a readable Word file")\n            logger.info("Report request %s completed: %s (%d bytes)",\n                        request_id, result.docx_path.name, result.docx_path.stat().st_size)\n            return result.markdown, str(result.docx_path)\n        except Exception as exc:\n            logger.exception("Report request %s failed", request_id)\n            return f"### Report could not be generated (reference `{request_id}`)\\n\\n{exc}", None\n\n    with gr.Blocks(title="SMS AC — SDLC specialist") as demo:\n        gr.Markdown("# SMS AC — LLM 2 SDLC specialist\\nUpload the software-design Word document, "\n                    "discuss missing decision factors, then create a detailed Word recommendation.")\n        design = gr.File(label="Software design (.docx)", file_types=[".docx"], type="filepath")\n        conversation = gr.Chatbot(label="Project discussion", type="messages", height=430)\n        history = gr.State([])\n        entry = gr.Textbox(label="Your message", placeholder="Tell me about your project or answer a clarification question")\n        send = gr.Button("Send")\n        send.click(on_message, [entry, conversation, history, design], [entry, conversation, history])\n        entry.submit(on_message, [entry, conversation, history, design], [entry, conversation, history])\n        produce = gr.Button("Generate detailed SDLC Word document", variant="primary")\n        report = gr.Markdown(label="Recommendation")\n        download = gr.File(label="Download Word document")\n        produce.click(on_report, [history, design], [report, download])\n    demo.queue(default_concurrency_limit=1).launch(share=share, server_name="0.0.0.0", show_error=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--adapter", type=Path, required=True)\n    parser.add_argument("--share", action="store_true")\n    args = parser.parse_args()\n    launch(args.adapter, args.share)\n'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
                'transformers==4.57.3', 'peft==0.17.1', 'accelerate==1.11.0',
                'bitsandbytes==0.48.2', 'safetensors==0.6.2',
                'python-docx==1.2.0', 'gradio==5.49.1'], check=True)
documents_path = PROJECT / 'src/sms_ac/documents.py'
documents_path.parent.mkdir(parents=True, exist_ok=True)
documents_path.write_text(DOCUMENTS_SOURCE, encoding='utf-8')
(PROJECT / 'src/sms_ac/__init__.py').write_text('', encoding='utf-8')
module_path = PROJECT / 'src/sms_ac/llm2_chat.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(MODULE_SOURCE, encoding='utf-8')
server_path = PROJECT / 'scripts/serve_llm2_chat.py'
server_path.parent.mkdir(parents=True, exist_ok=True)
server_path.write_text(SERVER_SOURCE, encoding='utf-8')
print('Chat application ready')


In [ ]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-u', str(PROJECT / 'scripts/serve_llm2_chat.py'),
                '--adapter', str(ADAPTER), '--share'], cwd=PROJECT, check=True)
